# Deal Quality Scorer Agent

**Community Contribution by dc_dalin**

Built an agent that scores deals on multiple dimensions (value, quality, urgency, description clarity) and gives an overall recommendation.

Uses Week 8 concepts: Agent base class, Structured Outputs, integration with ScannerAgent.

## Setup

In [2]:
import os
import sys
import logging
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import List

sys.path.insert(0, os.path.join(os.getcwd(), '..', '..'))

from agents.agent import Agent
from agents.deals import Deal, DealSelection
from agents.scanner_agent import ScannerAgent

load_dotenv(override=True)

root = logging.getLogger()
root.setLevel(logging.INFO)

## Quality Score Model

In [3]:
class QualityScore(BaseModel):
    value_score: int = Field(
        description="Score from 0-100 representing how good the price is compared to typical market value."
    )
    quality_score: int = Field(
        description="Score from 0-100 representing the product quality based on description, brand, specs."
    )
    urgency_score: int = Field(
        description="Score from 0-100 representing how time-sensitive this deal is."
    )
    description_score: int = Field(
        description="Score from 0-100 representing how detailed the product description is."
    )
    overall_score: int = Field(
        description="Overall score from 0-100 as weighted average of all scores."
    )
    recommendation: str = Field(
        description="One of: 'Excellent Deal - Act Now', 'Good Deal - Worth Considering', 'Fair Deal', 'Skip - Not Worth It'"
    )
    reasoning: str = Field(
        description="Brief 2-3 sentence explanation of the scores and recommendation"
    )

## Deal Quality Scorer Agent

In [4]:
class DealQualityScorerAgent(Agent):
    MODEL = "gpt-5-mini"

    SYSTEM_PROMPT = """You are an expert deal analyzer. Score deals on multiple dimensions.

Analyze each deal:
1. Value Score (0-100): Price vs typical market value
2. Quality Score (0-100): Product quality based on brand, specs, features
3. Urgency Score (0-100): Time sensitivity of the deal
4. Description Score (0-100): How detailed and informative the description is

Overall score: (value * 0.4) + (quality * 0.3) + (urgency * 0.15) + (description * 0.15)

Recommendations:
- 85-100: "Excellent Deal - Act Now"
- 70-84: "Good Deal - Worth Considering"
- 50-69: "Fair Deal"
- 0-49: "Skip - Not Worth It"
"""

    name = "Deal Quality Scorer"
    color = Agent.MAGENTA

    def __init__(self):
        self.log("Deal Quality Scorer Agent is initializing")
        self.openai = OpenAI()
        self.log("Deal Quality Scorer Agent is ready")

    def score_deal(self, deal: Deal) -> QualityScore:
        self.log(f"Analyzing deal: {deal.product_description[:50]}...")

        user_prompt = f"""Analyze this deal and provide quality scores:

Product: {deal.product_description}
Price: ${deal.price}
URL: {deal.url}"""

        response = self.openai.chat.completions.parse(
            model=self.MODEL,
            messages=[
                {"role": "system", "content": self.SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt},
            ],
            response_format=QualityScore,
            reasoning_effort="minimal",
        )

        result = response.choices[0].message.parsed
        self.log(f"Deal scored: {result.overall_score}/100 - {result.recommendation}")
        return result

    def score_multiple_deals(self, deals: List[Deal]) -> List[tuple[Deal, QualityScore]]:
        self.log(f"Scoring {len(deals)} deals")
        scored_deals = [(deal, self.score_deal(deal)) for deal in deals]
        scored_deals.sort(key=lambda x: x[1].overall_score, reverse=True)
        return scored_deals

    def format_score_report(self, deal: Deal, score: QualityScore) -> str:
        description = deal.product_description
        if len(description) > 150:
            description = description[:150] + "..."

        return f"""
{'=' * 80}
DEAL QUALITY REPORT
{'=' * 80}

Product: {description}
Price: ${deal.price}

SCORES:
  Value:       {score.value_score:3d}/100  {'█' * (score.value_score // 10)}
  Quality:     {score.quality_score:3d}/100  {'█' * (score.quality_score // 10)}
  Urgency:     {score.urgency_score:3d}/100  {'█' * (score.urgency_score // 10)}
  Description: {score.description_score:3d}/100  {'█' * (score.description_score // 10)}

  OVERALL:     {score.overall_score:3d}/100  {'█' * (score.overall_score // 10)}

RECOMMENDATION: {score.recommendation}

REASONING: {score.reasoning}

URL: {deal.url}
{'=' * 80}
"""

## Example

Test the UI with something like:

**Product Description:**
```
Apple MacBook Air M2 13.6-inch, 8GB RAM, 256GB SSD, 1080p FaceTime HD camera, Touch ID, works with iPhone/iPad
```

**Price:** 899

**URL:** https://www.bestbuy.com/macbook-air-m2

## Gradio UI

In [ ]:
import gradio as gr

scorer = DealQualityScorerAgent()

def score_deal_ui(product_desc, price, url):
    if not product_desc or not price:
        return "Please provide product description and price"
    
    try:
        deal = Deal(
            product_description=product_desc,
            price=float(price),
            url=url if url else "https://example.com"
        )
        
        score = scorer.score_deal(deal)
        
        result = f"""
### {score.recommendation}

**Overall Score: {score.overall_score}/100**

#### Score Breakdown:
- Value: {score.value_score}/100
- Quality: {score.quality_score}/100
- Urgency: {score.urgency_score}/100
- Description: {score.description_score}/100

#### Reasoning:
{score.reasoning}
"""
        return result
    except Exception as e:
        return f"Error: {str(e)}"

demo = gr.Interface(
    fn=score_deal_ui,
    inputs=[
        gr.Textbox(label="Product Description", lines=4, placeholder="Enter product details..."),
        gr.Number(label="Price ($)", value=0),
        gr.Textbox(label="URL (optional)", placeholder="https://...")
    ],
    outputs=gr.Markdown(label="Quality Score"),
    title="Deal Quality Scorer",
    description="Score deals based on value, quality, urgency, and description"
)

demo.launch()